# Adaptive Kalman on the bench (DAO)

Same bench interface as `DAOKalman.ipynb`, but the whole procedure is done by `AdaptiveKalmanController` in a single loop:

0. **Delay** (before the Kalman): a small random dither is added to the integrator commands, and the frame at which it shows up in `z` gives the loop delay. A wrong delay makes the Kalman diverge.
1. **Bootstrap**: during the first `window` frames, the controller runs its own integrator (same gain and leak as the bench) and records the pseudo-open-loop telemetry.
2. **Identification**: one AR(p) model per mode is fitted to that telemetry. Q is the variance of its one-step prediction errors (always positive, so no mode collapses to K = 0) and R is a fraction of Q. The steady-state gain is computed (DARE) and the model is validated.
3. **Kalman**: if the model is accepted, the controller switches to the Kalman command. If the innovation monitor (NIS) fires, it falls back to the integrator.

**Sign convention.** The controller assumes `u = leak * u + gain * z` and `y = z + u_{k-delay}`. The bench integrator is `u = leak * u - gain * z`, so the measurement is given to the controller as `SIGN * z`, with `SIGN = -sign(gain)`: the controller's integrator (gain `|gain|`) is then exactly the bench integrator. The gain of `z` with respect to the command, measured for each mode in step 1, is passed as `wfs_gain`: the Kalman uses `z / wfs_gain`, so that its pseudo-open-loop measurement is in command units.

**Without DAO** (e.g. on a laptop), `fake_dao.py` simulates the bench: the notebook runs unchanged, and the last cell compares with the true residual of the simulation.

In [ ]:
import time
import numpy as np
import torch
import pylab as plt

try:
    import dao
except ImportError:              # no DAO on this machine: simulated bench, see fake_dao.py
    import fake_dao
    bench = fake_dao.install(delay=2, sign=+1, wfs_gain=1.0)
    import dao
from kalman.controller import AdaptiveKalmanConfig, AdaptiveKalmanController

device = "cuda"

In [ ]:
MODAL_VECTOR_INFERENCE_SHM = dao.shm("/tmp/pyrModesNN.im.shm")    # WFS modal measurement z
CONTROL_SIGNAL_SHM = dao.shm("/tmp/ctrlModes07.im.shm")            # modal DM command u
SHM_LOOP_CMD_NN_SHM = dao.shm("/tmp/lpCmdNN.im.shm")               # loop closed flag
SHM_LOOP_GAIN_SHM = dao.shm("/tmp/lpGain.im.shm")
SHM_LOOP_LEAK_SHM = dao.shm("/tmp/lpLeak.im.shm")

shm = dao.shm('/tmp/lpCmdNNModalEnable.im.shm')
shm.set_data(np.full_like(np.asarray(shm.get_data()), 0))


def read_z(min_wait=5e-5):
    """
    Wait for a new WFS frame and return a copy of it, flattened.

    If Python fell behind (CUDA warm-up, identification, OS jitter), frames
    piled up in the semaphore and a read returns at once, with a frame that
    was already processed. Those reads are skipped: keep reading until one
    actually waits (> min_wait seconds) for a new frame.
    """
    while True:
        t0 = time.perf_counter()
        z = MODAL_VECTOR_INFERENCE_SHM.get_data(check=True, semNb=3)
        if time.perf_counter() - t0 > min_wait:
            return np.array(z, dtype=np.float32).reshape(-1)

In [ ]:
gain = float(SHM_LOOP_GAIN_SHM.get_data()[0, 0])
leak = float(SHM_LOOP_LEAK_SHM.get_data()[0, 0])
SIGN = -np.sign(gain)                    # bench: u = leak*u - gain*z  ->  controller sees SIGN*z

Nmodes = MODAL_VECTOR_INFERENCE_SHM.get_data().size
N_KF = 70                                # modes controlled by the Kalman; the others keep the bench integrator
U_MAX = 100                              # safety: open the loop if any command exceeds this

print(f"gain {gain}, leak {leak}, SIGN {SIGN:+.0f}, {Nmodes} measured modes, {N_KF} with Kalman")

## 1. Measure the delay

Integrator loop (as in `DAOKalman.ipynb`) on all modes, plus a random ±dither `p` on the first `N_KF` modes. The response of `z` to the dither at lag `j`, $\sum z_k p_{k-j} / \sum p^2$, is zero before the delay and $\pm 1$ at it. At the delay, `SIGN * response` is `-WFS_GAIN`: **-1** if `z` and `u` are in the same units. A smaller magnitude (WFS optical gain, NN scaling) would make the pseudo-open-loop measurement mostly command history, so the gain of each mode (`WFS_GAINS`, plotted) is passed to the controller, which divides `z` by it. The per-mode estimates are clipped to [0.5, 2] x the median of all modes, to limit the effect of noisy ones.

In [ ]:
N_DITHER = 3000
rng = np.random.default_rng(0)

control_signal = CONTROL_SIGNAL_SHM.get_data() * 0
cs = control_signal.reshape(-1)          # flat view: cs[i] is mode i
u = np.zeros(Nmodes, dtype=np.float32)
u_log, z_log, p_log = [], [], []

for k in range(N_DITHER):
    z = read_z()
    u = leak * u - gain * z[:Nmodes]
    p = np.zeros(N_KF, dtype=np.float32)
    if k < 500:                          # integrator only: gives the size of the commands
        u_log.append(u[:N_KF])
    else:
        if k == 500:
            amp = 0.2 * np.std(u_log, axis=0)
        p = (amp * rng.choice([-1, 1], N_KF)).astype(np.float32)
        z_log.append(z[:N_KF])
        p_log.append(p)
    cs[:Nmodes] = u
    cs[:N_KF] += p
    if np.abs(cs).max() > U_MAX:
        print(k, "out")
        break
    CONTROL_SIGNAL_SHM.set_data(control_signal)
CONTROL_SIGNAL_SHM.set_data(control_signal * 0)

Z, P = np.stack(z_log), np.stack(p_log)
response = np.array([(Z[j:] * P[:len(P) - j]).sum() / (P[:len(P) - j] ** 2).sum() for j in range(6)])
DELAY = int(np.argmax(np.abs(response)))
WFS_GAIN = float(-SIGN * response[DELAY])            # 1 when z and u are in the same units
P_d = P[:len(P) - DELAY]
gain_per_mode = -SIGN * (Z[DELAY:] * P_d).sum(0) / (P_d ** 2).sum(0)
WFS_GAINS = np.clip(gain_per_mode, 0.5 * np.median(gain_per_mode), 2 * np.median(gain_per_mode))

print("SIGN * response per lag:", (SIGN * response).round(2))
print(f"delay = {DELAY} frames, WFS_GAIN = {WFS_GAIN:.2f}, "
      f"per mode: 10-90% [{np.quantile(gain_per_mode, 0.1):.2f}, {np.quantile(gain_per_mode, 0.9):.2f}]")
plt.plot(gain_per_mode, ".", label="measured"); plt.plot(WFS_GAINS, "x", label="used")
plt.axhline(WFS_GAIN, color="k", ls=":"); plt.legend()
plt.xlabel("mode"); plt.ylabel("WFS gain"); plt.show()

## 2. Adaptive Kalman

In [ ]:
# The WFS gain was measured under the integrator. When the Kalman reduces the residual, the
# pyramid's optical gain rises: if the true gain is rho x wfs_gain, the loop rings at fs/4 for
# rho -> 2 (high NIS) and diverges above. Overestimating the gain is safe, so give it a margin.
GAIN_MARGIN = 2.0

cfg = AdaptiveKalmanConfig(
    n_modes=N_KF,
    wfs_gain=torch.tensor(GAIN_MARGIN * WFS_GAINS, dtype=torch.float32),
    delay=DELAY,                         # frames between the command and the measurement that sees it
    window=10000,                        # frames used to identify the model
    update_period=4000,                  # re-identify every 4000 frames
    max_updates=None,                    # keep re-identifying
    identification="prediction_error",   # Q from the one-step prediction errors: never collapses
    ar_order=1,                          # robust; 4 predicts better but needs an accurate wfs_gain
    R_fraction=0.1,                      # R = 0.1 * Q (low-noise WFS)
    integrator_gain=abs(gain),           # bootstrap and fallback = the bench integrator
    integrator_leak=leak,
    allow_integrator_fallback=True,      # back to the integrator on a breakdown...
    nis_threshold=10.0,                  # ... (NIS of 2-5 is normal when the WFS gain moves)
    nis_patience=2,                      # ... for 2 NIS windows in a row
    recover_on="new_model",              # back to the Kalman only with a new model
    ident_dtype=torch.float64,
    dare_dtype=torch.float64,
)
ctrl = AdaptiveKalmanController(cfg, device, torch.float32)

In [ ]:
N_FRAMES = 10000

control_signal = CONTROL_SIGNAL_SHM.get_data() * 0
cs = control_signal.reshape(-1)
u_hi = np.zeros(Nmodes - N_KF, dtype=np.float32)
CONTROL_SIGNAL_SHM.set_data(control_signal)

z_log, u_log, mode_log = [], [], []

print("waiting for the loop flag (lpCmdNN)...")
while not SHM_LOOP_CMD_NN_SHM.get_data()[0, 0]:
    read_z()

for k in range(N_FRAMES):
    if not SHM_LOOP_CMD_NN_SHM.get_data()[0, 0]:
        print(k, "loop opened from DAO")
        break

    z = read_z()

    # Kalman (or bootstrap integrator) on the first N_KF modes, bench integrator on the others
    u_kf = ctrl.rt_step(SIGN * torch.from_numpy(z[:N_KF]).to(device)).cpu().numpy()
    u_hi = leak * u_hi - gain * z[N_KF:Nmodes]
    cs[:N_KF] = u_kf
    cs[N_KF:Nmodes] = u_hi

    if np.abs(cs).max() > U_MAX:
        print(k, "out", ctrl.mode)
        break
    CONTROL_SIGNAL_SHM.set_data(control_signal)

    z_log.append(z[:N_KF])
    u_log.append(u_kf)
    mode_log.append(ctrl.mode)

    # Between frames: NIS monitor and identification (read_z skips the frames they make us miss)
    n_events = len(ctrl.history)
    ctrl.slow_path()
    for h in ctrl.history[n_events:]:
        print(k, h)

CONTROL_SIGNAL_SHM.set_data(control_signal * 0)
print("final mode:", ctrl.mode)

## 3. Results

RMS of the measured residual `z` on the Kalman modes (100-frame mean). The dotted line is the frame where the model was accepted.

In [ ]:
z_log, u_log, mode_log = np.stack(z_log), np.stack(u_log), np.array(mode_log)
rms = np.sqrt((z_log ** 2).mean(axis=1))
rms_mean = np.convolve(rms, np.ones(100) / 100, mode="valid")

plt.plot(rms_mean)
for h in ctrl.history:
    if h["event"] == "accepted":
        plt.axvline(h["frame"], color="k", ls=":")
plt.xlabel("frame"); plt.ylabel("RMS of z"); plt.yscale("log"); plt.show()

first = next((h["frame"] for h in ctrl.history if h["event"] == "accepted"), None)
if first is None:
    print("no model accepted:", [h.get("reasons") for h in ctrl.history])
else:
    print(f"integrator: {rms[first // 2:first].mean():.4g}   Kalman: {rms[first + 500:].mean():.4g}")
    print("spectral radius of A:", ctrl.check.spectral_radius)
    print("K diagonal:", torch.diagonal(ctrl.dare.K).cpu().numpy().round(2))

# Ringing at fs/4 while the Kalman runs: the WFS gain is underestimated (raise GAIN_MARGIN)
def fs4_fraction(z):
    F = np.abs(np.fft.rfft(z - z.mean(0), axis=0)) ** 2
    f = np.fft.rfftfreq(len(z))
    return F[(f > 0.2) & (f < 0.3)].sum() / F[1:].sum()
for m in ("integrator", "bootstrap", "kalman"):
    sel = mode_log == m
    if sel.sum() > 500:
        print(f"{m:10s}: {sel.sum():6d} frames, RMS of z {np.sqrt((z_log[sel] ** 2).mean()):.4g}, "
              f"power near fs/4 {100 * fs4_fraction(z_log[sel]):.0f}%")

In [ ]:
if ctrl.nis_log:
    f, nis = zip(*ctrl.nis_log)
    plt.plot(f, nis, "o-")
    plt.axhline(1, color="k", ls="--")
    plt.axhline(cfg.nis_threshold, color="0.5", ls=":")
    plt.yscale("log"); plt.xlabel("frame"); plt.ylabel("mean NIS / N_modes"); plt.show()

In [ ]:
# Everything needed to replay the run offline: dither (Z, P), telemetry (z as measured,
# u as sent by the controller, both on the Kalman modes), settings, events and NIS
import json
np.savez(
    "DAO_adaptive_run.npz",
    Z_dither=Z, P_dither=P, z=z_log, u=u_log, mode=mode_log, GAIN_MARGIN=GAIN_MARGIN,
    SIGN=SIGN, WFS_GAIN=WFS_GAIN, WFS_GAINS=WFS_GAINS, DELAY=DELAY, gain=gain, leak=leak, N_KF=N_KF,
    config=json.dumps({k: v for k, v in vars(cfg).items() if isinstance(v, (int, float, str, bool, type(None)))}),
    history=json.dumps(ctrl.history, default=float),
    nis_log=np.array(ctrl.nis_log, dtype=float).reshape(-1, 2),
)

if hasattr(dao, "bench"):        # fake bench only: the true residual is known
    print(dao.bench.summary(n_modes=N_KF))